# MLlib hands-on: RandomForest para predecir la severidad del retraso

## Descripción de las variables

El dataset está compuesto por las siguientes variables:

1. **Year** 2008
2. **Month** 1
3. **DayofMonth** 1-31
4. **DayOfWeek** 1 (Monday) - 7 (Sunday)
5. **DepTime** hora real de salida (local, hhmm)
6. **CRSDepTime** hora prevista de salida (local, hhmm)
7. **ArrTime** hora real de llegada (local, hhmm)
8. **CRSArrTime** hora prevista de llegada (local, hhmm)
9. **UniqueCarrier** código del aparato
10. **FlightNum** número de vuelo
11. **TailNum** identificador de cola: aircraft registration, unique aircraft identifier
12. **ActualElapsedTime** tiempo real invertido en el vuelo
13. **CRSElapsedTime** en minutos
14. **AirTime** en minutos
15. **ArrDelay** retraso a la llegada, en minutos: se considera que un vuelo ha llegado "on time" si aterrizó menos de 15 minutos más tarde de la hora prevista en el Computerized Reservations Systems (CRS).
16. **DepDelay** retraso a la salida, en minutos
17. **Origin** código IATA del aeropuerto de origen
18. **Dest** código IATA del aeropuerto de destino
19. **Distance** en millas
20. **TaxiIn** taxi in time, in minutes
21. **TaxiOut** taxi out time in minutes
22. **Cancelled** *si el vuelo fue cancelado (1 = sí, 0 = no)
23. **CancellationCode** razón de cancelación (A = aparato, B = tiempo atmosférico, C = NAS, D = seguridad)
24. **Diverted** *si el vuelo ha sido desviado (1 = sí, 0 = no)
25. **CarrierDelay** en minutos: El retraso del transportista está bajo el control del transportista aéreo. Ejemplos de sucesos que pueden determinar el retraso del transportista son: limpieza de la aeronave, daño de la aeronave, espera de la llegada de los pasajeros o la tripulación de conexión, equipaje, impacto de un pájaro, carga de equipaje, servicio de comidas, computadora, equipo del transportista, problemas legales de la tripulación (descanso del piloto o acompañante) , daños por mercancías peligrosas, inspección de ingeniería, abastecimiento de combustible, pasajeros discapacitados, tripulación retrasada, servicio de inodoros, mantenimiento, ventas excesivas, servicio de agua potable, denegación de viaje a pasajeros en mal estado, proceso de embarque muy lento, equipaje de mano no válido, retrasos de peso y equilibrio.
26. **WeatherDelay** en minutos: causado por condiciones atmosféricas extremas o peligrosas, previstas o que se han manifestado antes del despegue, durante el viaje, o a la llegada.
27. **NASDelay** en minutos: retraso causado por el National Airspace System (NAS) por motivos como condiciones meteorológicas (perjudiciales pero no extremas), operaciones del aeropuerto, mucho tráfico aéreo, problemas con los controladores aéreos, etc.
28. **SecurityDelay** en minutos: causado por la evacuación de una terminal, re-embarque de un avión debido a brechas en la seguridad, fallos en dispositivos del control de seguridad, colas demasiado largas en el control de seguridad, etc.
29. **LateAircraftDelay** en minutos: debido al propio retraso del avión al llegar, problemas para conseguir aterrizar en un aeropuerto a una hora más tardía de la que estaba prevista.

In [1]:
import os

from IPython.core.displaypub import DisplayPublisher
from databricks.connect import DatabricksSession
from dotenv import load_dotenv

# Initialize Spark session and environment variables
spark = DatabricksSession.builder.getOrCreate()

load_dotenv()
STORAGE_ACCOUNT_NAME = os.getenv("STORAGE_ACCOUNT_NAME")
AZURE_ACCOUNT_KEY = os.getenv("AZURE_ACCOUNT_KEY")

In [2]:
# Leemos el dataset de HDFS. Esta operación todavía no hace la lectura hace una pasada 
# sobre los datos para inferir el esquema
flightsDF = (spark.read.option(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", AZURE_ACCOUNT_KEY)\
                      .option("header", "true"))\
                      .option("inferSchema", "true")\
                      .csv(f"abfss://datos@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/flights-jan-apr-2018.csv")

In [3]:
flightsDF.printSchema()

root
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCity: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCity: string (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- Cancelled: double (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: double (nullable = true)
 |-- ActualElapsedTime: double (nullable = true)
 |-- AirTime: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDelay: double (nullable = true)
 |-- LateAircraftDelay: double (nullable = true)



Vamos a utilizar algunos transformadores habituales, y un algoritmo RandomForest que es un estimador. Utilizaremos:

* StringIndexer para convertir variables tipo String en variables categóricas pero cuyos valores son números reales con la parte decimal a 0, tal como necesitan los algoritmos de Spark.
* Bucketizer para discretizar la columna de ArrDelay sin dar nombre a las categorías, solo numeros. Será nuestra variable target.
* VectorAssembler para unir las columnas de las features en una sola de tipo vector
* RandomForest que es un estimador, como algoritmo de predicción de la severidad
* Pipeline, que es un estimador y que incluirá todos los elementos anteriores.

In [4]:
flightsDF.select("Origin").distinct().count()

356

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.ml.feature import Bucketizer, StringIndexer, VectorAssembler, StringIndexerModel
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

cleanFlightsDF = flightsDF.where("ArrDelay is not null and DepDelay is not null and DepTime is not null and ArrTime is not null")\
                          .withColumn("ArrDelay", F.col("ArrDelay").cast(IntegerType()))\
                          .withColumn("DepDelay", F.col("DepDelay").cast(IntegerType()))\
                          .withColumn("ArrTime", F.col("ArrTime").cast(IntegerType()))\
                          .withColumn("DepTime", F.col("DepTime").cast(IntegerType()))#.cache()

# Dividimos en train y test de manera aleatoria usando la semilla 123 para los números aleatorios, para que sea reproducible. 
# Esto devuelve una lista de dos DataFrames. La división hará que el primer elemento de la lista sea un DF 
# con aprox. el 70 % de las filas. Lo usaremos para entrenar. El otro DF tendrá aprox el 30 % restante de las filas. 

splits = cleanFlightsDF.randomSplit([0.7, 0.3], seed = 123)
trainDF = splits[0]#.cache() # El primer DF tiene el 70 % de los datos
testDF = splits[1]#.cache()

# Definimos tres categorías: <15, entre 15 y 60, >60. Cada una se codifica con un número real: 0.0, 1.0, 2.0 
splitsDelays = [-float("inf"), 15, 60, float("inf")]
arrDelayBucketizer = Bucketizer(splits=splitsDelays, inputCol="ArrDelay", outputCol="ArrDelayBucketed")

# Definimos varias franjas: 00:00 - 06:00, 06:00 - 12:00, 12:00 - 18:00, 18:00 - 22:00, 22:00 - 00:00
splitsDepTime = [0, 600, 1200, 1800, 2200, 2400]
depTimeBucketizer = Bucketizer(splits=splitsDepTime, inputCol="DepTime", outputCol="DepTimeBucketed")

# Indexamos las columnas categóricas Origin, Dest y DayOfWeek, para traducirlas a reales con la parte decimal a 0.
# Recordemos que esto también introduce metadatos adicionales en la columna resultante para indicar que, aunque sea
# una columna de números reales, en realidad están representando categorías y debe ser tratada como tal por el algoritmo
indexers = StringIndexer(inputCols=["Origin", "Dest", "DayOfWeek"], 
                         outputCols=["OriginIndexed", "DestIndexed", "DayOfWeekIndexed"],
                         handleInvalid="keep")  # si tuviéramos que transformar una categoría no vista, se codifica con uno más de la última existente

# Ahora unimos todas las columnas que se usarán como variables en una sola columna de tipo vector llamada featuresVector
vectorAssembler = VectorAssembler(inputCols = ["DepDelay", "DepTimeBucketed", "OriginIndexed", "DestIndexed", "DayOfWeekIndexed"], 
                                  outputCol = "featuresVector")

randomForest = RandomForestClassifier(featuresCol = "featuresVector",
                                      labelCol = "ArrDelayBucketed",
                                      numTrees = 50, maxBins=360)  # maxBins debe ser al menos el número de categorías de la variable que tenga más categorías

pipeline = Pipeline(stages=[
    arrDelayBucketizer,  # <-- Added here
    depTimeBucketizer,
    indexers,
    vectorAssembler,
    randomForest
])

Aplicamos fit para ajustar el pipeline completo. Lo que esto hace es, para cada etapa:
- Si la etapa es un Transformer, invoca al método transform() pasándole el DF que recibe de la etapa anterior,
  y envía el resultado (DF transformado) a la etapa siguiente del pipeline.
- Si la etapa es un Estimator, invoca al método fit() pasándole como argumento el DF recibido de la etapa anterior,
  y después invoca transform() sobre el objeto devuelto por fit, pasando de nuevo dicho DF como argumento. 
  Finalmente el DF devuelvo por transform es enviado a la etapa siguiente del pipeline.

In [6]:
trainDF.printSchema()

root
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCity: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCity: string (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- DepDelay: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- ArrDelay: integer (nullable = true)
 |-- Cancelled: double (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: double (nullable = true)
 |-- ActualElapsedTime: double (nullable = true)
 |-- AirTime: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDelay: double (nullable = true)
 |-- LateAircraftDelay: double (nullable = true)



In [7]:
#trainDF_sample = trainDF.sample(withReplacement=False, fraction=0.05, seed=42)
pipelineModel = pipeline.fit(trainDF)   # CUIDADO: esta línea tarda aproximadamente 15 minutos

#### Guardamos el pipeline entrenado en DBFS

El modelo entrenado no contiene datos y existe como un objeto de Python convencional. Al guardarlo, no se puede especificar directamente Azure DataLake Storage como destino, ya que la comunicación con ADLS se hace a través de Spark porque el cluster tiene habilitada la propiedad adecuada (mediante token) para permitirle la escritura, pero fuera de Spark no tenemos acceso excepto que hubiésemos montado un volumen en DBFS apuntando a una ubicación de ADLS.

In [10]:
# Point directly to the folder in your Volume where the files landed
model_path = "/Volumes/workspace/default/stages/flights_rf"

# Load the raw Spark model back into memory
pipelineModel.write().overwrite().save(model_path)

AnalysisException: [PATH_ALREADY_EXISTS] Path dbfs:/Volumes/workspace/default/stages/flights_rf\metadata already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.outputPathAlreadyExistsError(QueryCompilationErrors.scala:2950)
	at org.apache.spark.sql.execution.datasources.InsertIntoHadoopFsRelationCommand.run(InsertIntoHadoopFsRelationCommand.scala:146)
	at org.apache.spark.sql.execution.command.DataWritingCommandExec.$anonfun$sideEffectResult$5(commands.scala:140)
	at org.apache.spark.sql.execution.SparkPlan.runCommandInAetherOrSpark(SparkPlan.scala:201)
	at org.apache.spark.sql.execution.command.DataWritingCommandExec.$anonfun$sideEffectResult$4(commands.scala:140)
	at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:114)
	at com.databricks.spark.util.FrameProfilerExporter$.maybeExportFrameProfiler(FrameProfilerEx

#### Mostramos la estructura de carpetas guardada en DBFS

In [ ]:
# !ls -l /dbfs/user/flights_rf
# !ls -ls /dbfs/user/flights_rf/metadata
# !cat /dbfs/user/flights_rf/metadata/part-00000

In [ ]:
# !ls -ls /dbfs/user/flights_rf/stages
# !cat /dbfs/user/flights_rf/stages/0_Bucketizer_8d7ab4a234a8/metadata/part-00000
#
# !ls -l /dbfs/user/flights_rf/stages/4_RandomForestClassifier_2d56a1d93e72/data
#
# !ls -l /dbfs/user/flights_rf/stages/4_RandomForestClassifier_2d56a1d93e72/treesMetadata/

#### Contenido de la carpeta data: un fichero parquet

In [18]:
# tree_data = spark.read.parquet("/Volumes/workspace/default/stages/flights_rf\stages\4_RandomForestClassifier_c0df68d1456e/data/")
# tree_data.show()

#### Contenido de la carpeta treesMetadata: otro fichero parquet

In [19]:
# spark.read.parquet("dbfs:/user/flights_rf/stages/4_RandomForestClassifier_2d56a1d93e72/treesMetadata").display()

### Predicciones con el pipeline entrenado

Vamos a hacer predicciones sobre los datos de test. Esto devolverá un nuevo DF al que se le han añadido varias columnas al final:
1. **`rawPrediction`**: magnitud que tiene una interpretación diferente según el algoritmo pero que intuitivamente nos da una
  medida de la confianza en cada posible clase (cuanto mayor, más confianza se tiene en que esa es la clase del ejemplo). En
  nuestro caso será un vector de 3 números reales puesto que nuestro problema tiene 3 clases
2. **`probability`**: vector de probabilidades de que el ejemplo pertenezca a cada una de las 3 clases de nuestro problema
3. **`prediction`**: clase para la cual la rawProbability es mayor.

## Explicar qué ocurre cuando no viene la columna ArrDelay en el DataFrame de datos para predicción

En un conjunto de datos reales a los cuales tengamos que hacerles predicciones, lo normal es que no tengamos ninguna columna target (ArrDelay) porque desconocemos ese dato y precisamente por eso querríamos predecirlo. Sin embargo, en nuestro pipeline hemos incluido la etapa `arrDelayBucketizer` que discretiza la variable ArrDelay. Esto significa que cualquier DataFrame que queramos predecir (`transform(...)`) con el pipeline entrenado tiene que tener una columna llamada ArrDelay para que la pieza `arrDelayBucketizer` no reviente y pueda funcionar con normalidad, **a pesar de que dicha columna ArrDelay no va a ser utilizada por nadie más en el pipeline a la hora de predecir**, ya que naturalmente, el RandomForestClassificationModel (modelo ya entrenado) no necesita ninguna variable target para calcular predicciones. Por tanto tenemos dos opciones:
* Incluir una variable ArrDelay ficticia, inventada, en el DataFrame de datos para predecir, con cualquier valor. La finalidad es simplemente **evitar** que la pieza `arrDelayBucketizer` (y por tanto, el pipeline completo) reviente al hacer predicciones.
* Quitar del pipeline la pieza `arrDelayBucketizer` y ajustarla **por fuera del pipeline**, antes incluso de hacer la división en train y test. De esta manera, como el pipeline no contendrá ninguna pieza que vaya a requerir una columna ArrDelay, no será problema que el DataFrame con el que predecimos no incluya esa columna.

In [17]:
predictionsDF = pipelineModel.transform(testDF)

predictionsDF.select("rawPrediction", "probability", "prediction").where("prediction = 1.0").show(truncate=False)

#pipelineModel.transform(realDataDF.withColumn("ArrDelay", F.lit(999)))

+---------------------------------------------------------+------------------------------------------------------------+----------+
|rawPrediction                                            |probability                                                 |prediction|
+---------------------------------------------------------+------------------------------------------------------------+----------+
|[14.049784890892685,31.58306335427562,4.367151754831704] |[0.28099569781785366,0.6316612670855123,0.08734303509663406]|1.0       |
|[9.10299910825993,35.309282103640264,5.587718788099798]  |[0.18205998216519861,0.7061856420728054,0.11175437576199597]|1.0       |
|[8.957991963753136,34.99987966010336,6.042128376143498]  |[0.17915983927506274,0.6999975932020673,0.12084256752286997]|1.0       |
|[10.315772436682929,33.431213537797525,6.253014025519544]|[0.2063154487336586,0.6686242707559505,0.1250602805103909]  |1.0       |
|[8.337377834786802,35.55672942811705,6.105892737096135]  |[0.16674755669573

In [20]:
print("Hay {0} ejemplos de entrenamiento".format(trainDF.count()))

Hay 1706260 ejemplos de entrenamiento


In [21]:
predictionsDF.printSchema()

root
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCity: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCity: string (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- DepDelay: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- ArrDelay: integer (nullable = true)
 |-- Cancelled: double (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: double (nullable = true)
 |-- ActualElapsedTime: double (nullable = true)
 |-- AirTime: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDelay: double (nullable = true)
 |-- LateAircraftDelay: double (nullable = true)
 |-- ArrDelayBucketed: double (null

### Evaluación del modelo (evaluamos las predicciones que hemos hecho sobre el DF de test)

In [22]:
# Select (prediction, true label) and compute test error
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Objeto evaluador, para evaluar las predicciones. Compara una columna de predicciones con la columna target del verdadero valor
# Hay varias métricas posibles, pero nosotros hemos elegido la más simple: porcentaje de acierto en clasificación.
evaluator = MulticlassClassificationEvaluator(
    labelCol="ArrDelayBucketed", predictionCol="prediction", metricName="accuracy")

accuracy = evaluator.evaluate(predictionsDF)

print("Test Error = %g " % (1.0 - accuracy))

Test Error = 0.0807595 


In [23]:
pipelineModel.stages[-1]  # La última etapa del pipeline entrenado es el algoritmo Random Forest entrenado

RandomForestClassificationModel: uid=RandomForestClassifier_c0df68d1456e, numTrees=50, numClasses=3, numFeatures=5

In [24]:
pipelineModel.stages[-1].featureImportances

SparseVector(5, {0: 0.9851, 1: 0.011, 2: 0.0017, 3: 0.0023, 4: 0.0})

### Ajuste de híper-parámetros utilizando Cross Validation sobre el subconjunto de train

In [25]:
randomForest.getOrDefault("numTrees")

50

### ATENCIÓN: la siguiente celda tarda unos 23 minutos en ejecutar

In [26]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Objeto ParamGrid al que le tenemos que indicar cada uno de los parámetros sobre los que queremos buscar,
# y la lista de valores posibles que queremos probar con cada uno. Deben ser parámetros tomados directamente
# del objeto estimador que se añadió al pipeline (no pueden ser de ningún otro modelo). En nuestro caso este objeto
# está almacenado en la variable randomForest que habíamos creado en celdas anteriores.
paramGrid = ParamGridBuilder()\
    .addGrid(randomForest.numTrees, [50, 150])\
    .addGrid(randomForest.maxDepth, [3, 5])\
    .build() # como tenemos 2 híper-parámetros con 2 valores posibles cada uno, hay 4 combinaciones posibles

# CrossValidator es un estimador. Cuando invocamos a fit(), probará todas las combinaciones de valores de los 
# parámetros, invocando con cada combinación al método fit() del objeto pipeline que le hemos pasado como argumento
crossval = CrossValidator(estimator=pipeline,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

# Como hemos hecho 3 folds, habrá que entrenar 3 veces cada modelo candidato (cada combinación de parámetros)
# y obtener su media de accuracy. En total vamos a entrenar 12 modelos + 1 modelo extra al final 
# con todo el dataset de train (sin folds) usando la combinación de híper-parámetros que ya sabemos que funciona mejor

cvModel = crossval.fit(trainDF.sample(0.1, seed=123))  # intentamos evitar que tarde mucho en este ejemplo
cvModel.bestModel

AnalysisException: [NOT_SUPPORTED_WITH_SERVERLESS] PERSIST TABLE is not supported on serverless compute. SQLSTATE: 0A000

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at com.databricks.serverless.ServerlessGCEdgeCheck$.throwError(ServerlessGCEdgeCheck.scala:65)
	at com.databricks.serverless.ServerlessGCEdgeCheck$.checkBlockCacheCommand(ServerlessGCEdgeCheck.scala:43)
	at org.apache.spark.sql.connect.service.SparkConnectAnalyzeHandler.process(SparkConnectAnalyzeHandler.scala:323)
	at org.apache.spark.sql.connect.service.SparkConnectAnalyzeHandler.$anonfun$handle$3(SparkConnectAnalyzeHandler.scala:79)
	at org.apache.spark.sql.connect.service.SparkConnectAnalyzeHandler.$anonfun$handle$3$adapted(SparkConnectAnalyzeHandler.scala:71)
	at org.apache.spark.sql.connect.service.SessionHolder.$anonfun$withSession$2(SessionHolder.scala:633)
	at org.apache.spark.sql.SparkSession.withActive(SparkSession.scala:860)
	at org.apache.spark.sql.connect.service.SessionHolder.$anonfun$withSession

In [ ]:
prediccionesMejorDF = cvModel.bestModel.transform(testDF)

#### El objeto RandomForestModel (modelo ajustado presente en la última etapa del pipeline ya ajustado) dispone de ciertos atajos para recuperar valores de algunos parámetros (por ejemplo getNumTrees) pero no de otros (por ejemplo, no existe getMaxDepth)

In [ ]:
rf = cvModel.bestModel.stages[-1]
print("Número óptimo de árboles: {0}".format(rf.getNumTrees))
print("Max depth óptimo: {0}".format(rf.getOrDefault('maxDepth')))